# Notebook 00b · Sentinel-2 date selection — Hanoi & HCMC

Selects the 2018 acquisition dates that
`03_Transferability_Vietnam_S2_median.ipynb` composites for each city.
Screening only — no point sampling, no predictor table.

The study was designed around the **percentile** composite, the best of the four
predictors on Milan. Cells 1–3b establish what imagery actually exists over
Vietnam; Cell 4 is where that forces a change of method.

## Pipeline

| Cell | Does |
|---|---|
| 1 | Config |
| 2 | Build each city's 30 km AOI, check availability |
| 3 | Score every 2018 date on cloud / validity / coverage → CSV |
| 3b | **Inspect the scene tables** |
| 4 | **Decision point** — thresholds, method, date selection |
| 5 | Write `s2_extraction_metadata.json` per city |
| 6 | Seasonal check |

> Run order is linear. Cell 3 is slow and quota-heavy but cached.

## Cell 1 · Imports & configuration

In [ ]:
# # ── Interpreter check ────────────────────────────────────────────────────────
# # If this notebook stalls on the import cell below, it is almost always the
# # wrong kernel: this machine also has C:\Python312, which has no `ee`/`geemap`.
# import sys, os
# print('interpreter :', sys.executable)
# print('version     :', sys.version.split()[0])
# if os.path.join('IMD-Mapping', '.venv') not in sys.executable:
#     print('
# *** WRONG KERNEL -- select .venv (3.11.4) in the kernel picker ***')



In [ ]:
# When needed, authenticate and initialise GEE
import ee
# ee.Authenticate()   # run once
ee.Initialize(project='ee-xiaoo-503606')
print('GEE initialised.')

In [ ]:
# PROJ fix (mirrors notebooks 00/01b) -- must precede geopandas/rasterio import
import os
import pyproj
os.environ['PROJ_LIB']  = pyproj.datadir.get_data_dir()
os.environ['PROJ_DATA'] = pyproj.datadir.get_data_dir()

import json
import time
import warnings

import ee
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box

from s2_utils import (S2_BANDS, harmonize, cloud_mask, valid_mask,
                      USE_CLDPRB, CLDPRB_THRESH, min_dates_for)

warnings.filterwarnings('ignore')

# ── Source ────────────────────────────────────────────────────────────────────
GEE_PROJECT   = 'ee-xiaoo-503606'
S2_YEAR       = 2018
S2_COLLECTION = 'COPERNICUS/S2_SR_HARMONIZED'   # L2A. No silent fallback to TOA.

# ── Composite: what we are AIMING for ────────────────────────────────────────
# The Milan study's best predictor. Whether Vietnam can actually support it is
# decided in Cell 4, against the scene tables -- not assumed here.
TARGET_METHOD  = 'percentile'
PERCENTILES    = (10, 25, 50, 75, 90)
MIN_DATES_PCTL = min_dates_for(PERCENTILES)      # 17

# Fallback if the target proves unreachable. Median has no minimum date count,
# so it is defined for any stack depth.
FALLBACK_METHOD = 'median'

# ── Acceptance gates ─────────────────────────────────────────────────────────
MIN_DATES         = 3      # dates per city
MIN_OBS_PER_PIXEL = 2.5    # mean valid looks per AOI pixel -- rejects a 1-date
                           # "median", and a selection padded with partial
                           # swaths that the date count alone would wave through

# ── Cities ────────────────────────────────────────────────────────────────────
# Centres / UTM / bbox copied from notebook 02 so this screens exactly the AOI
# notebook 03 samples.
BBOX_HALF = 15000                # 30 km x 30 km
CITIES = {
    'Hanoi': {'centre_lonlat': (105.85, 21.03), 'utm_crs': 'EPSG:32648'},
    'HCMC':  {'centre_lonlat': (106.70, 10.78), 'utm_crs': 'EPSG:32648'},
}

# ── Scene thresholds (notebook 00 baseline; Cell 4 may relax) ────────────────
MAX_CLOUD_AOI = 20     # %  cloud/shadow/cirrus over the AOI
MIN_VALID_PCT = 90     # %  AOI pixels surviving the full validity mask
MIN_COVERAGE  = 95     # %  AOI area inside the scene footprint
QA_SCALE      = 60     # m  reduction scale for mask statistics

# ── GEE throttling (quota is rate-based, not per-request) ────────────────────
SCORE_BATCH           = 6
PAUSE_BETWEEN_BATCHES = 2.0

# ── Paths ─────────────────────────────────────────────────────────────────────
# Scene scoring is method-independent, so the CSV cache sits outside any
# per-method directory -- the method decision must not force a rescore.
SCENE_DIR = './samples_S2_vietnam'
os.makedirs(SCENE_DIR, exist_ok=True)

def city_dir(city, method):
    return f'./samples_S2_{method}_{city}'

def scene_csv(city):
    return f'{SCENE_DIR}/s2_scene_candidates_{city}_{S2_YEAR}.csv'

plt.rcParams.update({
    'figure.dpi': 130, 'font.size': 10,
    'axes.titlesize': 11, 'axes.labelsize': 10,
    'axes.spines.top': False, 'axes.spines.right': False,
})

print(f'{S2_COLLECTION} | {S2_YEAR} | {list(CITIES)} | '
      f'{2*BBOX_HALF/1000:.0f}x{2*BBOX_HALF/1000:.0f} km AOI')
print(f'target: {TARGET_METHOD}{tuple(sorted(PERCENTILES))} '
      f'-> {10*len(PERCENTILES)} bands, needs >= {MIN_DATES_PCTL} dates')
print(f'fallback if unreachable: {FALLBACK_METHOD} -> {len(S2_BANDS)} bands')
print(f'gates: >= {MIN_DATES} dates AND >= {MIN_OBS_PER_PIXEL} obs/pixel')
print(f'baseline thresholds: cloud<={MAX_CLOUD_AOI}%  '
      f'valid>={MIN_VALID_PCT}%  coverage>={MIN_COVERAGE}%'
      f'  (cldprb {"<%d" % CLDPRB_THRESH if USE_CLDPRB else "off"})')

---
## Cell 2 · AOI & availability

The bbox is built in UTM then converted to WGS84 — matching notebook 02, so
this screens exactly the area notebook 03 samples. A bbox built in degrees
would not be square in metres.

In [ ]:
def build_aoi(cfg):
    """30 km x 30 km ee.Geometry around a city centre, built in UTM."""
    lon, lat = cfg['centre_lonlat']
    centre = gpd.GeoDataFrame(
        geometry=gpd.points_from_xy([lon], [lat]), crs='EPSG:4326'
    ).to_crs(cfg['utm_crs'])
    cx, cy = centre.geometry.x[0], centre.geometry.y[0]
    bbox_utm = box(cx - BBOX_HALF, cy - BBOX_HALF,
                   cx + BBOX_HALF, cy + BBOX_HALF)
    bbox_4326 = gpd.GeoSeries([bbox_utm], crs=cfg['utm_crs']).to_crs('EPSG:4326').iloc[0]
    return ee.Geometry.Polygon([[list(c) for c in bbox_4326.exterior.coords]])


for city, cfg in CITIES.items():
    cfg['aoi_geom'] = build_aoi(cfg)
    area = cfg['aoi_geom'].area(1).getInfo()
    print(f'{city:6s} AOI area: {area/1e6:,.0f} km2')

# ── Availability check -- hard stop, mirroring notebook 00 cell 3 ─────────────
start = ee.Date.fromYMD(S2_YEAR, 1, 1)
end   = start.advance(1, 'year')

for city, cfg in CITIES.items():
    raw = (ee.ImageCollection(S2_COLLECTION)
           .filter(ee.Filter.bounds(cfg['aoi_geom']))
           .filter(ee.Filter.date(start, end)))
    n = raw.size().getInfo()
    print(f'{city:6s} {S2_COLLECTION} scenes in {S2_YEAR}: {n}')
    if n == 0:
        raise RuntimeError(
            f'No {S2_COLLECTION} scenes for {S2_YEAR} over {city}. '
            'Switch to COPERNICUS/S2_HARMONIZED (L1C/TOA) and note the '
            'radiometric difference, or widen the date window and accept the '
            'loss of year alignment with the 2018 IMD label. Not switching '
            'automatically.')
    # harmonize() makes the 2018 archive band-homogeneous -- see s2_utils.
    cfg['s2_all'] = raw.map(harmonize)

print('\nSR coverage available for both cities -- proceeding to scoring.')

---
## Cell 3 · Score every date  *(slow, quota-heavy)*

Ported from notebook 00 cell 5. All three statistics come from a **single**
`reduceRegion` per date, batched with backoff — GEE's aggregation quota is
rate-based, so more than that raises `Too many concurrent aggregations`.

Cached to CSV per city; delete the CSV to force a rescore.

In [ ]:
def mark_usable(tbl, max_cloud=None, min_valid=None, min_cover=None):
    """Return a copy with `usable` / `reason` under the given thresholds.

    Defined here, not in Cell 4, because the scene table is written to CSV with
    these columns already on it -- exactly as notebook 00 cell 5 does. Cell 4
    re-applies it at other thresholds; the CSV always holds the baseline verdict.
    """
    max_cloud = MAX_CLOUD_AOI if max_cloud is None else max_cloud
    min_valid = MIN_VALID_PCT if min_valid is None else min_valid
    min_cover = MIN_COVERAGE  if min_cover is None else min_cover

    t = tbl.copy()
    ok_cloud = t['AOI_cloud%'] <= max_cloud
    ok_valid = t['valid%']     >= min_valid
    ok_cover = t['coverage%']  >= min_cover
    t['usable'] = np.where(ok_cloud & ok_valid & ok_cover, 'YES', 'NO')

    # WHY a date failed is what makes the table actionable: a wall of 'cloud'
    # relaxes away at a looser threshold, a wall of 'coverage' never does.
    def reason(r):
        bad = []
        if r['AOI_cloud%'] > max_cloud: bad.append('cloud')
        if r['valid%']     < min_valid: bad.append('valid')
        if r['coverage%']  < min_cover: bad.append('coverage')
        return ','.join(bad)

    t['reason'] = t.apply(reason, axis=1)
    return t


def make_score_date(s2_all, aoi_geom):
    """Build a score_date closure bound to one city's collection and AOI."""

    def score_date(d):
        d      = ee.String(d)
        day    = ee.Date(d)
        subset = s2_all.filter(ee.Filter.date(day, day.advance(1, 'day')))

        # An empty per-date subset would mosaic to a 0-band image and break
        # select('B2'). Fall back to a fully-masked stand-in so the date still
        # scores (as 0% everything) instead of raising.
        empty  = (ee.Image.constant([0] * len(S2_BANDS) + [0])
                  .rename(S2_BANDS + ['SCL']).selfMask())
        mosaic = ee.Image(ee.Algorithms.If(subset.size().gt(0),
                                           subset.mosaic(), empty))

        # All three statistics in ONE reduceRegion over a 3-band image.
        # Every band is unmasked: reduceRegion(mean) SKIPS masked pixels, so a
        # masked band returns null for a fully-clouded scene rather than the 0%
        # it should report.
        stats = (cloud_mask(mosaic).rename('cloud')
                 .addBands(valid_mask(mosaic).rename('valid'))
                 .addBands(mosaic.select('B2').mask().unmask(0).rename('coverage'))
                 .reduceRegion(reducer=ee.Reducer.mean(), geometry=aoi_geom,
                               scale=QA_SCALE, bestEffort=True, maxPixels=1e9))

        def pct(key):
            v = stats.get(key)
            return ee.Number(ee.Algorithms.If(v, v, 0)).multiply(100)

        return ee.Feature(None, {
            'date':       d,
            'n_scenes':   subset.size(),
            'AOI_cloud':  pct('cloud'),
            'valid':      pct('valid'),
            'coverage':   pct('coverage'),
            'meta_cloud': subset.aggregate_mean('CLOUDY_PIXEL_PERCENTAGE'),
        })

    return score_date


def _is_throttle(exc):
    """Throttling surfaces as HttpError 429 OR ee.EEException -- catch both."""
    s = str(exc).lower()
    return ('concurrent aggregations' in s or 'too many' in s
            or 'quota' in s or 'rate limit' in s or '429' in s)


def score_city(city, cfg):
    """Ranked scene table for one city. Cached to CSV."""
    path = scene_csv(city)
    if os.path.exists(path):
        tbl = pd.read_csv(path)
        # Older cached tables predate the usable/reason columns -- add them
        # rather than forcing a rescore, which would burn the GEE quota.
        if 'usable' not in tbl.columns:
            tbl = mark_usable(tbl)
            tbl.to_csv(path, index=False)
            print(f'{city}: loaded {len(tbl)} scored dates from {path} '
                  '(cached; added usable/reason)')
        else:
            print(f'{city}: loaded {len(tbl)} scored dates from {path} (cached)')
        return tbl

    times = cfg['s2_all'].aggregate_array('system:time_start').getInfo()
    all_dates = sorted({pd.Timestamp(t, unit='ms').strftime('%Y-%m-%d')
                        for t in times})
    print(f'{city}: scoring {len(all_dates)} distinct dates '
          f'in batches of {SCORE_BATCH}...')

    scorer = make_score_date(cfg['s2_all'], cfg['aoi_geom'])
    rows, n_batches = [], int(np.ceil(len(all_dates) / SCORE_BATCH))

    for bi in range(0, len(all_dates), SCORE_BATCH):
        batch = all_dates[bi:bi + SCORE_BATCH]
        fc = ee.FeatureCollection(ee.List(batch).map(scorer))
        for attempt in range(6):
            try:
                rows.extend([f['properties'] for f in fc.getInfo()['features']])
                break
            except Exception as e:
                if _is_throttle(e) and attempt < 5:
                    wait = min(60, 8 * (2 ** attempt))
                    print(f'  batch {bi//SCORE_BATCH + 1}/{n_batches}: '
                          f'throttled, retry {attempt+1}/5 in {wait}s...',
                          flush=True)
                    time.sleep(wait)
                else:
                    raise
        else:
            raise RuntimeError(
                f'{city}: batch starting {batch[0]} still throttled after 5 '
                'retries. Lower SCORE_BATCH (try 4) and re-run.')
        print(f'  {len(rows)}/{len(all_dates)} dates scored', flush=True)
        time.sleep(PAUSE_BETWEEN_BATCHES)

    tbl = pd.DataFrame(rows).rename(columns={
        'AOI_cloud': 'AOI_cloud%', 'valid': 'valid%',
        'coverage': 'coverage%', 'meta_cloud': 'tile_cloud%'})
    for c in ['AOI_cloud%', 'valid%', 'coverage%', 'tile_cloud%']:
        tbl[c] = pd.to_numeric(tbl[c], errors='coerce').round(2)

    tbl = tbl.sort_values('valid%', ascending=False).reset_index(drop=True)
    tbl.insert(0, 'rank', tbl.index + 1)

    # Acceptance verdict travels WITH the table, as in notebook 00 cell 5.
    tbl = mark_usable(tbl)
    tbl.to_csv(path, index=False)
    print(f'  -> {path}')
    return tbl


# Column order matching notebook 00's printout.
SCENE_COLS = ['rank', 'date', 'AOI_cloud%', 'valid%', 'coverage%',
              'tile_cloud%', 'n_scenes', 'usable', 'reason']

scene_tables = {}
for city, cfg in CITIES.items():
    print('=' * 62)
    scene_tables[city] = score_city(city, cfg)
print('=' * 62)

for city, t in scene_tables.items():
    n = int((t['usable'] == 'YES').sum())
    print(f'{city:6s} {n} of {len(t)} dates usable under all three criteria.')
print('\nScoring complete.')

---
## Cell 3b · Inspect the scene tables

One row per date, ranked by `valid%`, with the `usable` / `reason` verdict at
the **baseline** thresholds — the same shape as `scene_table` in notebook 00.

| column | meaning |
|---|---|
| `AOI_cloud%` | cloud / shadow / cirrus over the AOI |
| `valid%` | full F-mask validity — also no-data, saturated, snow |
| `coverage%` | fraction of the AOI inside the scene footprint |
| `tile_cloud%` | whole-MGRS-tile cloud, **reported only** — misleading for a sub-tile AOI |
| `n_scenes` | scenes mosaicked for that date |
| `reason` | which criteria a rejected date failed |

**What to check here:**

1. **How many rows exist at all?** The target composite needs ≥ 17 dates. Milan
   has 71 over the same AOI size.
2. **The `reason` column.** `cloud` failures relax away at a looser threshold.
   `coverage` failures are partial swaths — no cloud setting recovers them.
3. **`coverage%` against `AOI_cloud%`.** Watch for rows reading ~0% cloud with
   single-digit coverage: cloud is measured over the sliver that exists, so a
   near-empty swath scores as pristine.

In [ ]:
# Hanoi -- all dates, ranked by valid%, with the baseline usable/reason verdict
scene_tables['Hanoi'][SCENE_COLS]

In [ ]:
# HCMC -- all dates, ranked by valid%, with the baseline usable/reason verdict
scene_tables['HCMC'][SCENE_COLS]

---
## Cell 4 · Decision point — method and thresholds

The tables above answer the question Cell 1 left open. This cell makes the call
and prints the evidence for it.

### The finding

The 2018 L2A archive holds **10 dates over Hanoi and 16 over HCMC**, against the
**17** that percentiles require. Milan has 71 over the same AOI size — L2A
coverage outside Europe is sparse in 2018, since systematic global production
started later.

The shortfall survives accepting *every* date unfiltered, so this is an
**archive limitation, not a threshold that can be relaxed**. The `ALL (ceiling)`
rung below is what makes that visible: it is the most this collection could ever
supply, and it is still short.

**→ The target `percentile` composite is unreachable. Falling back to `median`,
which has no minimum date count.**

### Why date count is the wrong measure anyway

A median needs several looks at the *same* pixel. A date contributes a usable
look to `valid%` of the AOI, so mean stack depth is `obs/pixel = sum(valid%)/100`
— reported beside each count below. Partial swaths inflate the count while
adding almost nothing to depth, which is why HCMC's 16 dates yield fewer looks
per pixel than you would guess.

This is also why the rungs relax **cloud** hard while holding **coverage** near
100%: per-pixel masking removes cloud from a full-coverage scene, but nothing
recovers data outside a swath.

### Alternative considered and rejected

Moving Vietnam to 2020+ restores full coverage and percentiles, but pairs that
imagery with a **2018** GHSL label — real error in two fast-urbanising cities.
Revisit if the thin composites prove limiting in notebook 03.

In [ ]:
# Threshold rungs, strictest first. Rung 99 accepts everything -- not a real
# option, but it shows the CEILING this collection could ever supply.
LADDER = [
    {'rung': 0,  'cloud':  20, 'valid': 90, 'cover': 95, 'label': 'baseline'},
    {'rung': 1,  'cloud':  35, 'valid': 90, 'cover': 95, 'label': 'cloud 20->35'},
    {'rung': 2,  'cloud':  35, 'valid': 80, 'cover': 95, 'label': 'valid 90->80'},
    {'rung': 3,  'cloud':  70, 'valid': 25, 'cover': 95, 'label': 'cloud<=70 valid>=25'},
    {'rung': 4,  'cloud':  70, 'valid': 25, 'cover': 80, 'label': 'as 3, cover>=80'},
    {'rung': 99, 'cloud': 100, 'valid':  0, 'cover':  0, 'label': 'ALL (ceiling)'},
]


def obs_per_pixel(tbl):
    """Mean valid observations per AOI pixel = sum(valid%)/100.

    What actually governs a median, and what a date count hides: a partial
    swath adds a row but almost no depth.
    """
    return float((tbl['valid%'] / 100).sum())


def passing(city, step):
    """(n_dates, obs_per_pixel) for one city at one rung."""
    t = mark_usable(scene_tables[city], step['cloud'], step['valid'],
                    step['cover'])
    u = t[t['usable'] == 'YES']
    return len(u), obs_per_pixel(u)


# ── 1. Ladder table: what each threshold setting yields ──────────────────────
rows = []
for step in LADDER:
    row = {'rung': step['rung'], 'thresholds': step['label'],
           'cloud<=': step['cloud'], 'valid>=': step['valid'],
           'cover>=': step['cover']}
    for city in CITIES:
        n, obs = passing(city, step)
        row[f'{city}_n'] = n
        row[f'{city}_obs'] = round(obs, 2)
    rows.append(row)
ladder_df = pd.DataFrame(rows)

print('Dates in 2018: ' +
      '  '.join(f'{c}={len(scene_tables[c])}' for c in CITIES) +
      '   (Milan, same AOI size: 71)\n')
print(ladder_df.to_string(index=False))

# ── 2. Can the TARGET method be built? ───────────────────────────────────────
# Test against the ceiling: if the most permissive setting cannot supply enough
# dates, no threshold can, and the limitation is the archive itself.
ceiling = {c: len(scene_tables[c]) for c in CITIES}
target_ok = all(n >= MIN_DATES_PCTL for n in ceiling.values())

print('\n' + '-' * 64)
print(f'  Target: {TARGET_METHOD}{tuple(sorted(PERCENTILES))} '
      f'needs >= {MIN_DATES_PCTL} dates')
print('-' * 64)
for c, n in ceiling.items():
    verdict = 'OK' if n >= MIN_DATES_PCTL else f'SHORT by {MIN_DATES_PCTL - n}'
    print(f'  {c:6s} {n:3d} dates exist in total  ->  {verdict}')

if target_ok:
    COMPOSITE_METHOD = TARGET_METHOD
    print(f'\n  => Target reachable. method={COMPOSITE_METHOD}')
else:
    COMPOSITE_METHOD = FALLBACK_METHOD
    print(f'\n  => Unreachable even accepting every date unfiltered, so no')
    print(f'     threshold change can help -- an ARCHIVE limitation.')
    print(f'     Falling back: method={COMPOSITE_METHOD} '
          f'({len(S2_BANDS)} bands, no minimum date count)')

# ── 3. Pick the strictest rung clearing both gates for both cities ──────────
chosen = None
for step in LADDER:
    if step['rung'] == 99:                     # ceiling row is diagnostic only
        continue
    if all(n >= MIN_DATES and obs >= MIN_OBS_PER_PIXEL
           for n, obs in (passing(c, step) for c in CITIES)):
        chosen = step
        break

print('\n' + '=' * 64)
if chosen is None:
    selected = {}
    print(f'  NO rung clears both gates ({MIN_DATES} dates, '
          f'{MIN_OBS_PER_PIXEL} obs/px) -- nothing will be written')
    print('=' * 64)
    for city in CITIES:
        best = max((passing(city, s) for s in LADDER if s['rung'] != 99),
                   key=lambda x: x[1])
        print(f'  {city:6s} best achievable: {best[0]} dates, {best[1]:.2f} obs/pixel')
else:
    print(f'  SELECTED  method={COMPOSITE_METHOD}  rung {chosen["rung"]}: '
          f'{chosen["label"]}')
    print(f'            cloud<={chosen["cloud"]} valid>={chosen["valid"]} '
          f'cover>={chosen["cover"]}')
    print('=' * 64)
    selected = {}
    for city in CITIES:
        # Re-mark at the chosen rung: the table now carries THAT verdict, not
        # the baseline one the CSV was written with.
        t = mark_usable(scene_tables[city], chosen['cloud'], chosen['valid'],
                        chosen['cover'])
        scene_tables[city] = t
        selected[city] = sorted(t.loc[t['usable'] == 'YES', 'date'])
        u = t[t['usable'] == 'YES']
        print(f'  {city:6s} {len(u):2d} dates  {obs_per_pixel(u):5.2f} obs/pixel  '
              f'({obs_per_pixel(u)/30*100:3.0f}% of Milan)  '
              f'mean valid {u["valid%"].mean():.1f}%')
    print('\n  Thin composites -- report obs/pixel with any metric from these.')

    for city in CITIES:
        print(f'\n{city}:')
        print(scene_tables[city][SCENE_COLS].to_string(index=False))

---
## Cell 5 · Write metadata

One `s2_extraction_metadata.json` per city, in the schema notebook 03 reads.
Band names follow whichever method Cell 4 settled on.

The JSON carries `method_rationale`, `obs_per_pixel` and `seasonal_caveat`, so
the constraints travel with the data to anyone who reads the file without this
notebook.

In [ ]:
# Band names follow the method Cell 4 settled on: median returns the 10 raw
# bands unsuffixed (s2_utils._reduce_median), percentile returns band x pctl.
if COMPOSITE_METHOD == 'percentile':
    BAND_NAMES = [f'{b}_p{p}' for b in S2_BANDS for p in sorted(PERCENTILES)]
else:
    BAND_NAMES = list(S2_BANDS)

if not selected:
    print('Nothing written: Cell 4 found no qualifying rung.')
else:
    for city, dates in selected.items():
        outdir = city_dir(city, COMPOSITE_METHOD)
        os.makedirs(outdir, exist_ok=True)
        t = scene_tables[city]
        u = t[t['usable'] == 'YES']

        meta = {
            'city':            city,
            'centre_lonlat':   list(CITIES[city]['centre_lonlat']),
            'utm_crs':         CITIES[city]['utm_crs'],
            'bbox_half_m':     BBOX_HALF,
            'collection':      S2_COLLECTION,
            'year':            S2_YEAR,
            'selected_dates':  dates,
            'forced':          False,
            'method':          COMPOSITE_METHOD,
            'percentiles':     (list(sorted(PERCENTILES))
                                if COMPOSITE_METHOD == 'percentile' else None),
            'bands':           BAND_NAMES,
            'raw_bands':       list(S2_BANDS),
            'fill_rate_pct':   None,
            'reflectance_scale': 'raw DN (0-10000), unscaled',
            'cldprb_gate':     f'MSK_CLDPRB < {CLDPRB_THRESH}' if USE_CLDPRB else 'OFF',
            'thresholds': {
                'MAX_CLOUD_AOI': chosen['cloud'],
                'MIN_VALID_PCT': chosen['valid'],
                'MIN_COVERAGE':  chosen['cover'],
                'QA_SCALE':      QA_SCALE,
            },
            'escalation_rung':  chosen['rung'],
            'escalation_label': chosen['label'],

            # Composite depth -- the number to report with any metric.
            'obs_per_pixel':       round(obs_per_pixel(u), 2),
            'obs_per_pixel_milan': 30,

            # Why this method, decided in Cell 4 against the scene tables.
            'target_method':    TARGET_METHOD,
            'method_rationale': (
                f'{TARGET_METHOD}{tuple(sorted(PERCENTILES))} needs '
                f'{MIN_DATES_PCTL} dates; {S2_COLLECTION} holds only '
                f'{len(t)} over this AOI in {S2_YEAR}. The shortfall survives '
                'accepting every date unfiltered, so it is an archive '
                f'limitation, not a relaxable threshold. Fell back to '
                f'{COMPOSITE_METHOD}, which has no minimum date count. '
                'Rejected alternative: Vietnam at 2020+ restores percentiles '
                'but pairs that imagery with a 2018 GHSL label.'
            ) if COMPOSITE_METHOD != TARGET_METHOD else
                f'{TARGET_METHOD} reachable: {len(t)} dates available.',

            'months_with_imagery': sorted(
                pd.to_datetime(t['date']).dt.month.unique().tolist()),
            'months_composited': sorted(
                pd.to_datetime(pd.Series(dates)).dt.month.unique().tolist()),
            'seasonal_caveat': (
                'Dry-season composite: several months carry no 2018 L2A '
                'imagery at all over this AOI. Do not report as annual.'),

            'per_date_stats': u[['date', 'AOI_cloud%', 'valid%',
                                 'coverage%', 'n_scenes']].to_dict('records'),
            'source_notebook': '00b_S2_Extraction_Vietnam_2018.ipynb',
        }

        path = f'{outdir}/s2_extraction_metadata.json'
        with open(path, 'w') as f:
            json.dump(meta, f, indent=2)
        print(f'{city:6s} {len(dates)} dates, {len(BAND_NAMES)} bands, '
              f'{meta["obs_per_pixel"]:.2f} obs/px  ->  {path}')

    print(f'\nNext: 03_Transferability_Vietnam_S2_{COMPOSITE_METHOD}.ipynb')
    print(f'Milan reference: outputs_S2_{COMPOSITE_METHOD}/best_model_RF_S2.joblib')

---
## Cell 6 · Seasonal check

Which months the archive actually covers. With whole months missing, the
"median 2018 reflectance" is really the median of whatever season exists —
the evidence behind the dry-season caveat.

In [ ]:
fig, axes = plt.subplots(len(CITIES), 1, figsize=(12, 2.4 * len(CITIES)),
                         sharex=True)
axes = np.atleast_1d(axes)

for ax, city in zip(axes, CITIES):
    t = scene_tables[city]
    sel = t[t['usable'] == 'YES']
    ax.scatter(pd.to_datetime(t['date']).dt.dayofyear, t['valid%'],
               s=30, c='#bdc3c7', label=f'all ({len(t)})', linewidths=0)
    if len(sel):
        ax.scatter(pd.to_datetime(sel['date']).dt.dayofyear, sel['valid%'],
                   s=44, c='#27ae60', label=f'selected ({len(sel)})', linewidths=0)
    ax.set_ylabel('valid %')
    ax.set_ylim(-3, 105)
    ax.legend(fontsize=7, loc='lower right')
    ax.set_title(
        f'{city} — {len(sel)} dates, {obs_per_pixel(sel):.2f} obs/pixel'
        if len(sel) else f'{city} — no selection ({len(t)} dates exist)',
        fontweight='bold')

months = pd.date_range(f'{S2_YEAR}-01-01', f'{S2_YEAR}-12-01', freq='MS')
axes[-1].set_xticks([d.dayofyear for d in months])
axes[-1].set_xticklabels([d.strftime('%b') for d in months])
axes[-1].set_xlabel('day of year')
axes[-1].set_xlim(0, 366)

fig.suptitle(f'Sentinel-2 {S2_YEAR} availability — {" & ".join(CITIES)}',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{SCENE_DIR}/fig_date_selection.png', dpi=150, bbox_inches='tight')
plt.show()

# Month coverage of everything that EXISTS -- empty months cannot be recovered
# by any screening choice.
print(f'Months with any {S2_YEAR} imagery:')
for city in CITIES:
    t = scene_tables[city]
    counts = pd.to_datetime(t['date']).dt.month.value_counts().sort_index()
    print(f'  {city:6s} ' + ' '.join(
        f'{pd.Timestamp(S2_YEAR, m, 1).strftime("%b")}:{counts.get(m, 0)}'
        for m in range(1, 13)))
    empty = [pd.Timestamp(S2_YEAR, m, 1).strftime('%b')
             for m in range(1, 13) if m not in counts.index]
    if empty:
        print(f'         EMPTY ({len(empty)}/12): {", ".join(empty)}')

if selected:
    print('\nMonths composited:')
    for city, dates in selected.items():
        mm = sorted(pd.to_datetime(pd.Series(dates)).dt.month.unique())
        print(f'  {city:6s} ' + ', '.join(
            pd.Timestamp(S2_YEAR, m, 1).strftime('%b') for m in mm))
    print('\n=> DRY-SEASON composite. Do not report as annual.')